# ПО НОВОЙ

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random
import warnings
import os

plt.style.use('dark_background')
print("Рабочая директория:", os.getcwd())

# Игнорируем предупреждения для чистоты вывода
warnings.filterwarnings('ignore')

print("Начинаем выполнение Лабораторной работы №2...")

Начинаем выполнение Лабораторной работы №2...


In [ ]:
# ==========================================
# 1. ЗАГРУЗКА ИЗОБРАЖЕНИЙ
# ==========================================

# Путь к папке с 8 фотографиями
data_path = 'D:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_2\Pictures'

# Список для хранения изображений
images = []
images_gray = []

# Загружаем 8 изображений (имена: 1.jpg, 2.jpg ... 8.jpg)
for i in range(1, 9):
    filename = f"{i+1}.jpg"
    filepath = os.path.join(data_path, filename)
    
    if os.path.exists(filepath):
        img = cv2.imread(filepath)
        # Переводим в RGB для вывода
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Переводим в Grayscale для математики
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        images.append(img_rgb)
        images_gray.append(img_gray)
        print(f"Загружено: {filename}")
    else:
        print(f"Не найдено: {filename}. Проверьте путь.")

# Визуализация загруженных фото
if len(images) == 8:
    plt.figure(figsize=(20, 7))
    for i in range(8):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i])
        plt.title(f"Image #{i+1}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Загружено только {len(images)} изображений. Продолжаем с тем, что есть.")

In [5]:
# ==========================================
# БЛОК 1: ОПРЕДЕЛЕНИЕ ВСЕХ ФУНКЦИЙ
# ==========================================

def convolution(image, window):
    """Ручная свертка изображения с ядром"""
    # Вычисляем отступ (половина размера окна)
    start = int((window.shape[0] - 1) / 2)
    result = np.zeros(image.shape)
    
    # Проходим по изображению, избегая краев
    for i in range(start, image.shape[0] - start):
        for j in range(start, image.shape[1] - start):
            # Извлекаем фрагмент и выполняем поэлементное умножение
            fragment = image[i - start : i + start + 1, j - start : j + start + 1]
            result[i, j] = np.sum(fragment * window)
    return result

def local_max(e, window_size):
    """Поиск локальных максимумов в карте отклика 'e'"""
    start = int((window_size - 1) / 2)
    corner = dict()
    
    # Проходим по внутренним пикселям
    for i in range(start, e.shape[0] - start):
        for j in range(start, e.shape[1] - start):
            # Вырезаем окно вокруг пикселя
            x = e[i - start : i + start + 1, j - start : j + start + 1]
            
            # Если текущий пиксель является максимумом в окне
            if e[i, j] == np.max(x):
                corner[e[i, j]] = [j, i] # Сохраняем координаты [x, y]
    return corner

def corner_detect(image_gray, n_corners, window_size):
    """
    Детектор ключевых точек (углов).
    Возвращает массив координат найденных углов.
    """
    # 1. Сглаживание для уменьшения шума
    gauss_img = cv2.GaussianBlur(image_gray, (5, 5), 3)
    
    # 2. Вычисление градиентов (изменение яркости по X и Y)
    dy, dx = np.gradient(gauss_img)
    
    # 3. Окно усреднения для матрицы структуры
    window = np.ones((window_size, window_size))
    
    # 4. Вычисление компонентов матрицы структуры (c_xx, c_yy, c_xy)
    c_xx = convolution(dx * dx, window)
    c_yy = convolution(dy * dy, window)
    c_xy = convolution(dx * dy, window)
    
    # 5. Вычисление отклика (собственных чисел)
    e = np.zeros(gauss_img.shape)
    
    for i in range(gauss_img.shape[0]):
        for j in range(gauss_img.shape[1]):
            # Матрица 2x2 для текущего пикселя
            M = np.array([[c_xx[i, j], c_xy[i, j]], 
                          [c_xy[i, j], c_yy[i, j]]])
            
            # Находим собственные числа
            eigenvalues, _ = np.linalg.eig(M)
            
            # Критерий Харриса (упрощенный): берем минимальное собственное число
            e[i, j] = np.min(eigenvalues)
            
    # 6. Поиск локальных максимумов
    corner_dict = local_max(e, window_size)
    
    # 7. Сортировка и выбор топ-N лучших углов
    corners = []
    # sorted вернет список ключей (значений отклика) от большего к меньшему
    sorted_keys = sorted(corner_dict.keys(), reverse=True)
    
    for key in sorted_keys:
        corners.append(corner_dict[key])
        if len(corners) >= n_corners:
            break
            
    return np.array(corners)

def ncc_match(img1, img2, c1, c2, R):
    """
    Вычисление Normalized Cross-Correlation (NCC) для двух точек.
    c1, c2 - координаты точек [x, y] на img1 и img2.
    R - радиус окна сопоставления.
    """
    sum1 = 0
    sum2 = 0
    matching_score = 0
    
    # Собираем значения пикселей в окне вокруг точки
    mean1 = []
    mean2 = []
    
    for i in range(-R, R + 1):
        for j in range(-R, R + 1):
            # Проверка границ изображения, чтобы не выйти за пределы
            y1, x1 = c1[1] + i, c1[0] + j
            y2, x2 = c2[1] + i, c2[0] + j
            
            if 0 <= y1 < img1.shape[0] and 0 <= x1 < img1.shape[1] and \
               0 <= y2 < img2.shape[0] and 0 <= x2 < img2.shape[1]:
                mean1.append(img1[y1, x1])
                mean2.append(img2[y2, x2])
            else:
                # Если вышли за границу, добавляем 0 (или можно пропустить)
                mean1.append(0)
                mean2.append(0)

    w1_m = np.mean(mean1)
    w2_m = np.mean(mean2)
    
    # Вычисляем суммы квадратов отклонений (знаменатель)
    for k in range(len(mean1)):
        sum1 += (mean1[k] - w1_m)**2
        sum2 += (mean2[k] - w2_m)**2
        
    denom = np.sqrt(sum1 * sum2)
    
    if denom == 0:
        return 0

    # Вычисляем скалярное произведение (числитель)
    for k in range(len(mean1)):
        w1 = (mean1[k] - w1_m)
        w2 = (mean2[k] - w2_m)
        matching_score += w1 * w2
        
    # Нормализуем
    matching_score = matching_score / denom
    return matching_score

def naive_matching(img1, img2, corners1, corners2, R, NCCth):
    """
    Поиск пар точек между двумя изображениями.
    Возвращает список кортежей: [(точка1_img1, точка1_img2), ...]
    """
    matching = []
    ncc_matrix = np.zeros([corners1.shape[0], corners2.shape[0]])
    
    # Перебираем все возможные пары точек
    for i in range(corners1.shape[0]):
        for j in range(corners2.shape[0]):
            score = ncc_match(img1, img2, corners1[i], corners2[j], R)
            ncc_matrix[i, j] = score
            
            # Если корреляция выше порога, считаем точки совпавшими
            if score > NCCth:
                matching.append((corners1[i], corners2[j]))
                
    return matching

def match_model(matches):
    """
    Оценивает параметры модели преобразования (аффинная трансформация) 
    на основе найденных пар точек.
    Использует метод случайных выборок (RANSAC-подобный подход).
    """
    if len(matches) < 3:
        return None # Недостаточно точек для модели
        
    # Создаем матрицы для решения системы уравнений AX = B
    # Модель: x' = ax + by + c, y' = dx + ey + f
    # Итого 6 параметров: [a, b, c, d, e, f]
    A_full = np.zeros((2 * len(matches), 6))
    B_full = np.zeros((2 * len(matches), 1))
    
    for i in range(len(matches)):
        n1, n2 = matches[i] # n1 - точка на img1, n2 - точка на img2
        
        # Строка для координаты X
        A_full[2 * i, 0], A_full[2 * i, 1], A_full[2 * i, 4] = n1[0], n1[1], 1
        B_full[2 * i] = n2[0]
        
        # Строка для координаты Y
        A_full[2 * i + 1, 2], A_full[2 * i + 1, 3], A_full[2 * i + 1, 5] = n1[0], n1[1], 1
        B_full[2 * i + 1] = n2[1]
        
    best_err = float('inf')
    best_m = None
    
    # Итерации случайного выбора 3 точек для построения модели
    # (Упрощенная версия RANSAC из методички)
    iterations = 500 
    for _ in range(iterations):
        if len(matches) < 3: break
        
        # Берем случайные 3 пары точек
        sample_indices = random.sample(range(len(matches)), 3)
        sample_matches = [matches[idx] for idx in sample_indices]
        
        # Формируем матрицы для выборки
        A_sub = np.zeros((2 * len(sample_matches), 6))
        B_sub = np.zeros((2 * len(sample_matches), 1))
        
        for i in range(len(sample_matches)):
            n1, n2 = sample_matches[i]
            A_sub[2 * i, 0], A_sub[2 * i, 1], A_sub[2 * i, 4] = n1[0], n1[1], 1
            B_sub[2 * i] = n2[0]
            
            A_sub[2 * i + 1, 2], A_sub[2 * i + 1, 3], A_sub[2 * i + 1, 5] = n1[0], n1[1], 1
            B_sub[2 * i + 1] = n2[1]
            
        # Решаем систему (находим псевдообратную матрицу)
        try:
            A_inv = np.linalg.pinv(A_sub)
            solution = A_inv @ B_sub
            
            # Проверяем ошибку на ВСЕХ точках (not just sample)
            predicted_B = A_full @ solution
            error = np.sum((B_full - predicted_B)**2) / len(matches)
            
            if error < best_err:
                best_err = error
                best_m = solution
        except:
            continue
            
    return best_m

In [6]:
# ==========================================
# БЛОК 2: ВЫПОЛНЕНИЕ ОДОМЕТРИИ
# ==========================================

# Параметры (можно настроить)
NUM_CORNERS = 30       # Количество ключевых точек на изображении
WINDOW_SIZE = 5        # Размер окна для детектора углов
R_RADIUS = 15          # Радиус окна для NCC (чем больше, тем точнее, но медленнее)
NCC_THRESHOLD = 0.7    # Порог корреляции (от 0 до 1)

print("Шаг 1: Поиск ключевых точек на всех изображениях...")
all_corners = []
for idx, img in enumerate(images_gray):
    corners = corner_detect(img, n_corners=NUM_CORNERS, window_size=WINDOW_SIZE)
    all_corners.append(corners)
    print(f" - Image {idx+1}: найдено {len(corners)} точек.")

print("\nШаг 2: Сопоставление и построение траектории...")

# Массив для хранения траектории (x, y)
trajectory = [[0, 0]] 
current_x, current_y = 0, 0

# Проходим по парам соседних изображений (0-1, 1-2, ..., 6-7)
for i in range(len(images_gray) - 1):
    print(f"Обработка пары: Image {i+1} -> Image {i+2}")
    
    img1 = images_gray[i]
    img2 = images_gray[i+1]
    corners1 = all_corners[i]
    corners2 = all_corners[i+1]
    
    # 1. Находим пары точек
    matches = naive_matching(img1, img2, corners1, corners2, R=R_RADIUS, NCCth=NCC_THRESHOLD)
    print(f"   Найдено совпадений: {len(matches)}")
    
    if len(matches) >= 3:
        # 2. Оцениваем модель движения
        model_params = match_model(matches)
        
        if model_params is not None:
            # Извлекаем параметры трансформации
            # Модель: x' = ax + by + c, y' = dx + ey + f
            # Параметры: [a, b, c, d, e, f]
            # c - сдвиг по X (tx), f - сдвиг по Y (ty)
            tx = model_params[2][0]
            ty = model_params[5][0]
            
            # Обновляем текущую позицию камеры
            # Мы накапливаем сдвиги, чтобы построить полную траекторию
            current_x += tx
            current_y += ty
            trajectory.append([current_x, current_y])
            print(f"   Сдвиг камеры: dx={tx:.2f}, dy={ty:.2f}")
        else:
            print("   Ошибка: не удалось построить модель.")
            trajectory.append([current_x, current_y]) # Сохраняем предыдущую точку
    else:
        print("   Мало совпадений для построения модели.")
        trajectory.append([current_x, current_y])

Шаг 1: Поиск ключевых точек на всех изображениях...


NameError: name 'images_gray' is not defined

In [ ]:
# ==========================================
# БЛОК 3: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ==========================================

trajectory = np.array(trajectory)

plt.figure(figsize=(15, 5))

# 1. График траектории
plt.subplot(1, 2, 1)
plt.plot(trajectory[:, 0], trajectory[:, 1], 'o-', color='blue', linewidth=2, markersize=10)
plt.title("Camera Trajectory (Visual Odometry)")
plt.xlabel("X displacement")
plt.ylabel("Y displacement")
plt.grid(True)
# Добавим подписи номеров кадров
for j, txt in enumerate(range(1, len(trajectory)+1)):
    plt.annotate(txt, (trajectory[j, 0], trajectory[j, 1]), textcoords="offset points", xytext=(5,-5), ha='center')

# 2. Пример сопоставления точек (для отчета)
# Возьмем первую пару (0 и 1)
plt.subplot(1, 2, 2)
# Создаем объединенное изображение для наглядности
h1, w1 = images_gray[0].shape
h2, w2 = images_gray[1].shape
combined_h = max(h1, h2)
combined_w = w1 + w2 + 50 # отступ
combined_img = np.zeros((combined_h, combined_w), dtype=np.uint8)
combined_img[:h1, :w1] = images_gray[0]
combined_img[:h2, w1+50:] = images_gray[1]

plt.imshow(combined_img, cmap='gray')

# Рисуем линии сопоставления для первых 10 пар (чтобы не загромождать)
sample_matches = naive_matching(images_gray[0], images_gray[1], all_corners[0], all_corners[1], R=R_RADIUS, NCCth=NCC_THRESHOLD)
for k, (pt1, pt2) in enumerate(sample_matches[:10]):
    x1, y1 = pt1
    x2, y2 = pt2[0] + w1 + 50, pt2[1] # Сдвигаем координаты второй точки
    plt.plot([x1, x2], [y1, y2], 'r-', linewidth=1)
    plt.plot([x1, x2], [y1, y2], 'go', markersize=3)

plt.title(f"Point Matching Example (Img 1 vs Img 2)\nFound: {len(sample_matches)} pairs")
plt.axis('off')

plt.tight_layout()
plt.show()

print("\nГотово! Траектория построена.")